In [30]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

In [ ]:
# === 0. Setup ===
import os
import math
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import List, Tuple, Dict
from datetime import timedelta
import glob

# Optional (for torch Dataset skeleton – 학습 단계에서 유용)
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset
    TORCH_AVAILABLE = True
except Exception:
    TORCH_AVAILABLE = False
    print("[Info] PyTorch not installed. You can still run up to fold-splitting.")

@dataclass
class Config:
    lookback: int = 28   # L
    horizon: int  = 7    # H
    patch_len: int = 7   # for PatchTST later
    stride: int    = 1   # for PatchTST later
    n_splits: int  = 5   # K-fold
    embargo_days: int = 35  # purge gap around validation (≈ lookback + horizon)
    seed: int = 42

CFG = Config()
np.random.seed(CFG.seed)

from pathlib import Path
import json, shutil
ART_DIR = Path("./optuna_trial_results"); ART_DIR.mkdir(parents=True, exist_ok=True)


print(CFG)


Config(lookback=28, horizon=7, patch_len=7, stride=1, n_splits=5, embargo_days=35, seed=42)


In [32]:
# === 1. Load & sort ===
TRAIN_PATH = "./dataset/train.csv" 

df = pd.read_csv(TRAIN_PATH)

# Basic parsing
# date to datetime
df['date'] = pd.to_datetime(df['date'])

# sort by store_menu and date
df = df.sort_values(['store_menu', 'date']).reset_index(drop=True)
df = df.drop(['store', 'menu'], axis=1)


print("Rows:", len(df))
print("Unique store_menu:", df['store_menu'].nunique())
print(df.head(3))


Rows: 102676
Unique store_menu: 193
   date_ordinal       date          store_menu  sales
0        738521 2023-01-01  느티나무 셀프BBQ_1인 수저세트      0
1        738522 2023-01-02  느티나무 셀프BBQ_1인 수저세트      0
2        738523 2023-01-03  느티나무 셀프BBQ_1인 수저세트      0


In [33]:
# Feature engineering
"""
- adding only the calender features
- final output : df_feat
"""

def add_calendar_features(frame: pd.DataFrame) -> pd.DataFrame:
    f = frame.copy()
    f['dow'] = f['date'].dt.weekday           # 0..6
    f['dom'] = f['date'].dt.day               # 1..31
    f['month'] = f['date'].dt.month           # 1..12
    f['is_weekend'] = (f['dow'] >= 5).astype(int)

    # Sine/Cos encoding for weekly seasonality
    f['dow_sin'] = np.sin(2 * np.pi * f['dow'] / 7)
    f['dow_cos'] = np.cos(2 * np.pi * f['dow'] / 7)
    return f

df_feat = add_calendar_features(df)

# 사용할 입력 피처 목록 (sales + calendar)
FEATURE_COLS = ['sales', 'dow', 'dom', 'month', 'is_weekend', 'dow_sin', 'dow_cos']
TARGET_COL = 'sales'

print("Feature columns:", FEATURE_COLS)
df_feat.head(3)


Feature columns: ['sales', 'dow', 'dom', 'month', 'is_weekend', 'dow_sin', 'dow_cos']


,date_ordinal,date,store_menu,sales,dow,dom,month,is_weekend,dow_sin,dow_cos
0,738521,2023-01-01,느티나무 셀프BBQ_1인 수저세트,0,6,1,1,1,-0.781831,0.62349
1,738522,2023-01-02,느티나무 셀프BBQ_1인 수저세트,0,0,2,1,0,0.000000,1.00000
2,738523,2023-01-03,느티나무 셀프BBQ_1인 수저세트,0,1,3,1,0,0.781831,0.62349


In [34]:
# === 3. Sliding windows (28 -> 7) indexing ===
def build_samples_index(
    frame: pd.DataFrame,
    lookback: int,
    horizon: int,
    feature_cols: List[str],
    target_col: str = 'sales'
) -> pd.DataFrame:
    
    rows = []
    sample_id = 0

    for sm, g in frame.groupby('store_menu', sort=False):
        g = g.sort_values('date').reset_index(drop=True)
        n = len(g)
        # last valid target start index (inclusive)
        last_t = n - horizon
        # first valid target start index (input must fully exist)
        first_t = lookback
        for t in range(first_t, last_t + 1):
            inp_start = t - lookback
            inp_end   = t - 1
            tgt_start = t
            tgt_end   = t + horizon - 1

            rows.append({
                'sample_id': sample_id,
                'store_menu': sm,
                'input_start_idx': inp_start,
                'target_start_idx': tgt_start,
                'input_start_date': g.loc[inp_start, 'date'],
                'input_end_date':   g.loc[inp_end,   'date'],
                'target_start_date':g.loc[tgt_start, 'date'],
                'target_end_date':  g.loc[tgt_end,   'date'],
            })
            sample_id += 1

    samples = pd.DataFrame(rows).sort_values(['store_menu','target_start_date']).reset_index(drop=True)
    return samples

# ID mapping
SM_LIST = df_feat['store_menu'].drop_duplicates().tolist()
SM2ID = {sm:i+1 for i, sm in enumerate(SM_LIST)}  # 0은 UNK
UNK_ID = 0

samples = build_samples_index(df_feat, CFG.lookback, CFG.horizon, FEATURE_COLS, TARGET_COL)
print("Total samples:", len(samples))
samples.head(3)

def future_cal_features(start_date, H=7):
    dates = pd.date_range(start_date, periods=H, freq='D')
    dow = dates.weekday
    dow_sin = np.sin(2*np.pi*dow/7)
    dow_cos = np.cos(2*np.pi*dow/7)
    is_weekend = (dow>=5).astype(int)
    # H×3 -> 평평하게(간단)
    return np.concatenate([dow_sin, dow_cos, is_weekend], axis=0).astype(np.float32)  # len=H*3



# Optional: PyTorch dataset skeleton for later training
if TORCH_AVAILABLE:
    class SalesWindowDataset(Dataset):
        def __init__(self, base_df, samples_df, feature_cols, target_col, lookback, horizon):
            self.base = base_df
            self.samples = samples_df.reset_index(drop=True)
            self.feat_cols = feature_cols
            self.tgt_col = target_col
            self.L, self.H = lookback, horizon
            self.group = {sm:g.reset_index(drop=True) for sm,g in self.base.groupby('store_menu', sort=False)}

        def __len__(self): return len(self.samples)

        def __getitem__(self, idx):
            s = self.samples.iloc[idx]
            g = self.group[s['store_menu']]
            inp_start = int(s['input_start_idx']); inp_end = inp_start + self.L
            tgt_start = int(s['target_start_idx']); tgt_end = tgt_start + self.H

            X = g.loc[inp_start:inp_end-1, self.feat_cols].to_numpy(np.float32)   # (L,C)
            y = g.loc[tgt_start:tgt_end-1, self.tgt_col].to_numpy(np.float32)     # (H,)

            # store_menu id
            sm_id = SM2ID.get(s['store_menu'], UNK_ID)

            # 타깃 시작일 기준 미래 달력 피처
            fut_cal = future_cal_features(s['target_start_date'], H=self.H)        # (H*3,)

            return torch.from_numpy(X), torch.from_numpy(y), torch.tensor(sm_id, dtype=torch.long), torch.from_numpy(fut_cal)




Total samples: 96114


In [35]:
# k-fold validation (for each store_menu)

def build_time_kfold_splits_by_series(
    samples: pd.DataFrame,
    n_splits: int,
    lookback: int,
    horizon: int,
    embargo_days: int,
    verbose: bool = True,
):

    # Keep original row index so we can map local rows back to the global `samples`
    s = samples.sort_values(['store_menu', 'target_start_date']).reset_index(drop=False)
    s.rename(columns={'index': '_orig_idx'}, inplace=True)

    # Containers for global folds
    fold_tr = [set() for _ in range(n_splits)]
    fold_va = [set() for _ in range(n_splits)]

    # Process each univariate series separately
    for sm, g in s.groupby('store_menu', sort=False):
        g = g.sort_values('target_start_date').reset_index(drop=True)

        # 1) Unique candidate validation dates inside this series
        uniq_dates = pd.Series(g['target_start_date'].unique()).sort_values().to_list()

        # 2) Use only dates after warm-up
        warmup_days = lookback + horizon + embargo_days
        earliest_val_date = (
            pd.Timestamp(uniq_dates[0]) + pd.Timedelta(days=warmup_days)
            if len(uniq_dates) > 0 else None
        )
        val_date_candidates = (
            [d for d in uniq_dates if d >= earliest_val_date]
            if earliest_val_date else []
        )

        # Determine usable local splits for this series
        local_splits = min(n_splits, max(1, len(val_date_candidates))) if val_date_candidates else 1
        bins = (
            np.array_split(np.array(val_date_candidates), local_splits)
            if val_date_candidates else [np.array([], dtype='datetime64[ns]')]
        )

        # Assign local folds into global fold ids [0..n_splits-1]
        for k in range(n_splits):
            if k >= len(bins):
                # This series has fewer local bins than n_splits
                continue

            val_dates = bins[k]
            if len(val_dates) == 0:
                # No validation for this fold in this series
                continue

            val_start = pd.Timestamp(val_dates[0])
            val_end   = pd.Timestamp(val_dates[-1])

            # Local row positions for val/train within this series
            val_mask = g['target_start_date'].isin(val_dates)
            val_loc  = g.index[val_mask].to_numpy()

            cutoff = val_start - pd.Timedelta(days=embargo_days)
            train_mask = g['target_end_date'] < cutoff
            train_loc  = g.index[train_mask].to_numpy()

            if len(train_loc) == 0 or len(val_loc) == 0:
                # Not enough data given warm-up/embargo
                continue

            # Map back to original `samples` indices
            val_idx_global = g.loc[val_loc, '_orig_idx'].to_numpy()
            trn_idx_global = g.loc[train_loc, '_orig_idx'].to_numpy()

            fold_tr[k].update(trn_idx_global.tolist())
            fold_va[k].update(val_idx_global.tolist())

            if verbose:
                continue  # keep exact original control flow (skip printing section)

    # Convert sets to sorted numpy arrays
    folds = []
    for k in range(n_splits):
        tr = np.array(sorted(fold_tr[k]), dtype=int)
        va = np.array(sorted(fold_va[k]), dtype=int)
        if len(tr) > 0 and len(va) > 0:
            folds.append((tr, va))
            if verbose:
                print(f"[Fold {len(folds)}/{n_splits}] train_size={len(tr):,}, val_size={len(va):,}")
        else:
            if verbose:
                print(f"[Skip] Fold {k+1}: train={len(tr)}, val={len(va)}")

    # If too few valid folds, relax embargo and retry
    if len(folds) <= 1 and embargo_days > 0:
        if verbose:
            print("[Info] Too few valid folds. Relaxing embargo and retrying.")
        relaxed = max(lookback, embargo_days // 2)
        return build_time_kfold_splits_by_series(
            samples, n_splits, lookback, horizon, relaxed, verbose
        )

    return folds


# Build folds
folds = build_time_kfold_splits_by_series(
    samples=samples,
    n_splits=CFG.n_splits,
    lookback=CFG.lookback,
    horizon=CFG.horizon,
    embargo_days=CFG.embargo_days,  # e.g., 35
    verbose=True
)

if TORCH_AVAILABLE:
    full_dataset = SalesWindowDataset(
        base_df=df_feat,
        samples_df=samples,
        feature_cols=FEATURE_COLS,
        target_col=TARGET_COL,
        lookback=CFG.lookback,
        horizon=CFG.horizon
    )
    tr_idx, va_idx = folds[0]
    print(f"First fold -> train:{len(tr_idx)}, val:{len(va_idx)}")


[Fold 1/5] train_size=5,597, val_size=16,598
[Fold 2/5] train_size=22,195, val_size=16,598
[Fold 3/5] train_size=38,793, val_size=16,598
[Fold 4/5] train_size=55,391, val_size=16,405
[Fold 5/5] train_size=71,796, val_size=16,405
First fold -> train:5597, val:16598


In [36]:
# === Loss utilities (stable sMAPE′ and combined loss) ===
import torch.nn.functional as F

def charbonnier(x: torch.Tensor, delta: float = 1e-2) -> torch.Tensor:
    """Smooth |x| ≈ sqrt(x^2 + delta^2) to stabilize gradients."""
    return torch.sqrt(x * x + delta * delta)

def smape_prime(y_hat: torch.Tensor, y: torch.Tensor, delta: float = 1e-1, eps: float = 1e-3) -> torch.Tensor:
    """
    Stable sMAPE':
      2*|e| / (|y| + |y_hat| + eps)
    where |.| is Charbonnier to avoid zero-denominator spikes.
    """
    e = y_hat - y
    num = 2.0 * charbonnier(e, delta)                           # (B,H)
    den = charbonnier(y_hat, delta) + charbonnier(y, delta) + eps
    return (num / den).mean()

def mae_loss(y_hat: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    return torch.mean(torch.abs(y_hat - y))

def mse_loss(y_hat: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    return torch.mean((y_hat - y) ** 2)

def normalize_weights(a: float, b: float, c: float) -> tuple[float, float, float]:
    s = max(a + b + c, 1e-8)
    return a / s, b / s, c / s

class CombinedLoss(nn.Module):
    """
    L = a*MAE + b*MSE + c*sMAPE′, with a+b+c=1 (normalized internally).
    Compute on the ORIGINAL scale (after denorm).
    """
    def __init__(self, a: float = 0.34, b: float = 0.33, c: float = 0.33,
                 smape_delta: float = 1e-1, smape_eps: float = 1e-3):
        super().__init__()
        self.a, self.b, self.c = normalize_weights(a, b, c)
        self.delta = smape_delta
        self.eps = smape_eps

    def forward(self, y_hat: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        mae = mae_loss(y_hat, y)
        mse = mse_loss(y_hat, y)
        sp  = smape_prime(y_hat, y, delta=self.delta, eps=self.eps)
        return self.a * mae + self.b * mse + self.c * sp


In [37]:
# === PatchTST (Original CI-style) + RevIN ===
import math
import torch
import torch.nn as nn

class RevIN(nn.Module):
    """Per-sample, per-channel normalization with reversible denorm."""
    def __init__(self, eps: float = 1e-5, min_std: float = 1.0):
        super().__init__()
        self.eps = eps
        self.min_std = min_std

    def forward(self, x, sales_ch: int = 0, stats=None, mode='norm'):
        # x: (B, L, C)
        if mode == 'norm':
            mu = x.mean(dim=1, keepdim=True)                 # (B,1,C)
            sigma = x.std(dim=1, keepdim=True) + self.eps    # (B,1,C)
            sigma = torch.clamp(sigma, min=self.min_std)
            x_n = (x - mu) / sigma
            mu_s = mu[:, :, sales_ch]                        # (B,1)
            sg_s = sigma[:, :, sales_ch]                     # (B,1)
            return x_n, (mu_s, sg_s)
        elif mode == 'denorm':
            mu_s, sg_s = stats                                # (B,1), (B,1)
            return x * sg_s + mu_s                           # x: (B,H) or (B,T,H)
        else:
            raise ValueError("mode must be 'norm' or 'denorm'")

class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding, length-agnostic."""
    def __init__(self, d_model: int, max_len: int = 1024):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.cos(pos * div)
        pe[:, 1::2] = torch.sin(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))          # (1, max_len, d_model)

    def forward(self, x):
        # x: (B, P, d)
        P = x.size(1)
        return x + self.pe[:, :P, :]

class PatchTST_OrigCI(nn.Module):
    def __init__(self,
                 lookback: int,
                 horizon: int,
                 c_in: int,
                 d_model: int = 256,
                 n_heads: int = 8,
                 depth: int = 3,
                 patch_len: int = 16,
                 stride: int = 8,
                 dropout: float = 0.1,
                 sales_ch: int = 0,
                 target_channels: list | None = None):
        super().__init__()
        assert lookback >= patch_len, "lookback must be >= patch_len"
        self.L = lookback
        self.H = horizon
        self.C = c_in
        self.patch_len = patch_len
        self.stride = stride
        self.sales_ch = sales_ch
        self.target_channels = target_channels  # e.g., [0] to predict only 'sales'

        # Number of patches per channel
        self.P = 1 + (self.L - self.patch_len) // self.stride

        # Patch embedding: Linear over patch_len (shared across channels)
        self.value_embedding = nn.Linear(self.patch_len, d_model)

        # Transformer encoder (shared across channels)
        enc = nn.TransformerEncoderLayer(d_model=d_model,
                                         nhead=n_heads,
                                         dim_feedforward=4*d_model,
                                         dropout=dropout,
                                         batch_first=True)
        self.encoder = nn.TransformerEncoder(enc, num_layers=depth)
        self.posenc = PositionalEncoding(d_model, max_len=max(1024, self.P + 8))

        # Channel-independent head: d_model -> H (shared across channels)
        self.head = nn.Linear(d_model, self.H)

        # Normalization wrapper
        self.revin = RevIN(eps=1e-5, min_std=1.0)

    def _patchify_unfold(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (B, L, C)
        return: (B*C, P, patch_len)
        """
        B, L, C = x.shape
        # (B, C, L)
        x_chw = x.permute(0, 2, 1)
        # Unfold over time: (B, C, P, patch_len)
        patches = x_chw.unfold(dimension=2, size=self.patch_len, step=self.stride)
        B, C, P, K = patches.shape
        return patches.contiguous().view(B * C, P, K)

    def forward(self, x: torch.Tensor):
        """
        x: (B, L, C)
        returns:
          - y_hat_n: (B, H) if single target channel, else (B, T, H) for T target channels
          - stats: (mu_s, sg_s) for denorm of the primary target channel (sales_ch)
        """
        B, L, C = x.shape
        # RevIN normalize
        x_n, stats = self.revin(x, sales_ch=self.sales_ch, mode='norm')

        # Patchify per channel and embed
        xp = self._patchify_unfold(x_n)                   # (B*C, P, patch_len)
        z = self.value_embedding(xp)                      # (B*C, P, d)
        z = self.posenc(z)                                # add PE
        z = self.encoder(z)                               # (B*C, P, d)
        z = z[:, -1, :]                                   # last-token pooling -> (B*C, d)

        # Channel-independent head, then reshape back to (B, C, H)
        y_all = self.head(z).view(B, C, self.H)           # (B, C, H)

        # Select target channels
        if self.target_channels is None:
            # Return all channels
            return y_all, stats
        else:
            y_sel = y_all[:, self.target_channels, :]     # (B, T, H)
            if y_sel.shape[1] == 1:
                y_sel = y_sel.squeeze(1)                  # (B, H)
            return y_sel, stats


In [38]:
# === Training utilities for PatchTST_OrigCI ===
from torch.utils.data import DataLoader, Subset

SELECT_SALES_ONLY = True   # True: use only sales channel for OrigCI
SALES_CH = 0               # index of sales channel in X[..., C]

def unpack_to_device(batch, device):
    """
    Accept (X,y) or (X,y,*) batches. Return X,y on device.
    Optionally slice to sales channel for OrigCI.
    """
    if isinstance(batch, (list, tuple)):
        X, y = batch[0], batch[1]
    else:
        raise ValueError("Unexpected batch format")
    X = X.to(device)
    y = y.to(device)
    if SELECT_SALES_ONLY:
        if X.dim() != 3:
            raise ValueError("Expect X as (B,L,C)")
        # keep only sales channel as (B,L,1)
        if X.size(-1) != 1:
            X = X[:, :, SALES_CH:SALES_CH+1]
    return X, y

@torch.no_grad()
def evaluate_origci(model, loader, loss_fn, device):
    model.eval()
    tot, n = 0.0, 0
    for batch in loader:
        X, y = unpack_to_device(batch, device)
        y_hat_n, stats = model(X)                                 # normalized space
        # y_hat_n is (B,H) if target_channels=[0], else (B,T,H)
        if y_hat_n.dim() == 3 and y_hat_n.size(1) == 1:
            y_hat_n = y_hat_n.squeeze(1)                          # (B,H)
        y_hat = model.revin(y_hat_n, stats=stats, mode='denorm')   # original scale
        loss = loss_fn(y_hat, y)
        bs = y.size(0); tot += loss.item() * bs; n += bs
    return tot / max(n, 1)

def train_one_epoch_origci(model, loader, optimizer, loss_fn, device, grad_clip=1.0):
    model.train()
    tot, n = 0.0, 0
    for batch in loader:
        X, y = unpack_to_device(batch, device)
        y_hat_n, stats = model(X)
        if y_hat_n.dim() == 3 and y_hat_n.size(1) == 1:
            y_hat_n = y_hat_n.squeeze(1)
        y_hat = model.revin(y_hat_n, stats=stats, mode='denorm')
        loss = loss_fn(y_hat, y)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        if grad_clip is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()

        bs = y.size(0); tot += loss.item() * bs; n += bs
    return tot / max(n, 1)

class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.best = float('inf')
        self.wait = 0
        self.stop = False
    def step(self, value):
        if value + self.min_delta < self.best:
            self.best = value; self.wait = 0
        else:
            self.wait += 1
            if self.wait >= self.patience:
                self.stop = True

def make_loaders_from_folds(dataset, folds, fold_id: int, batch_size: int = 256, num_workers: int = 0):
    tr_idx, va_idx = folds[fold_id]
    tr_ds = Subset(dataset, tr_idx)
    va_ds = Subset(dataset, va_idx)
    tr_loader = DataLoader(tr_ds, batch_size=batch_size, shuffle=True,  drop_last=False, num_workers=num_workers)
    va_loader = DataLoader(va_ds, batch_size=batch_size, shuffle=False, drop_last=False, num_workers=num_workers)
    return tr_loader, va_loader


In [39]:
# === Optuna objective for PatchTST_OrigCI + CombinedLoss ===
import optuna

def build_origci_and_optim(trial, lookback: int, horizon: int, c_in_raw: int, device: torch.device):
    # Model hparams
    d_model   = trial.suggest_categorical("d_model", [128, 192, 256, 320, 384, 512])
    n_heads   = trial.suggest_categorical("n_heads", [4, 8])
    depth     = trial.suggest_int("depth", 2, 4)
    patch_len = trial.suggest_categorical("patch_len", [4, 6, 7, 8, 12, 16])
    stride    = trial.suggest_categorical("stride", [1, 2, 4])
    dropout   = trial.suggest_float("dropout", 0.0, 0.3)

    # Channel setting for OrigCI
    c_in = 1 if SELECT_SALES_ONLY else c_in_raw
    tgt_ch = [0]  # predict sales channel only

    model = PatchTST_OrigCI(
        lookback=lookback, horizon=horizon, c_in=c_in,
        d_model=d_model, n_heads=n_heads, depth=depth,
        patch_len=patch_len, stride=stride, dropout=dropout,
        sales_ch=0, target_channels=tgt_ch
    ).to(device)

    lr        = trial.suggest_float("lr", 1e-4, 3e-3, log=True)
    if trial.suggest_categorical("wd_is_zero", [True, False]):
        weight_decay = 0.0
    else:
        weight_decay = trial.suggest_float("weight_decay_pos", 1e-8, 1e-3, log=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    return model, optimizer

# === Optuna objective (parameterized; no globals) ===
def objective_origci_enhanced(
    trial, *, device, lookback, horizon, c_in_raw, dataset, folds, art_dir=ART_DIR, stage="stage1"
):
    # ---- weights ----
    a = trial.suggest_float("w_mae", 0.0, 1.0)
    b = trial.suggest_float("w_mse", 0.0, 1.0)
    c = trial.suggest_float("w_smape", 0.0, 1.0)
    a, b, c = normalize_weights(a, b, c)

    # ---- stability ----
    smape_delta = trial.suggest_float("smape_delta", 5e-3, 3e-1, log=True)
    smape_eps   = trial.suggest_float("smape_eps",   1e-4, 5e-2, log=True)

    # ---- model hparams ----
    batch_size = trial.suggest_categorical("batch_size", [128, 256, 512])
    
    # Stage별 epochs 설정
    if stage == "stage1":
        max_epochs = trial.suggest_int("epochs", 15, 25)
    else:  # stage2
        max_epochs = trial.suggest_int("epochs", 70, 140)

    model, optimizer = build_origci_and_optim(
        trial, lookback=lookback, horizon=horizon, c_in_raw=c_in_raw, device=device
    )
    tr_loader, va_loader = make_loaders_from_folds(dataset, folds, fold_id=0, batch_size=batch_size)

    criterion = CombinedLoss(a=a, b=b, c=c, smape_delta=smape_delta, smape_eps=smape_eps)
    stopper = EarlyStopping(patience=8 if stage == "stage2" else 5)  # stage2에서는 더 오래 기다림
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs)

    # trial artifact dir
    tdir = art_dir / f"{stage}_trial_{trial.number:03d}"; tdir.mkdir(exist_ok=True, parents=True)
    ckpt_path = tdir / "best.pt"; cfg_path = tdir / "config.json"

    # config to reproduce (기존 + trial params + loss value)
    cfg = {
        "lookback": lookback, "horizon": horizon,
        "c_in_raw": c_in_raw, "select_sales_only": SELECT_SALES_ONLY, "sales_ch": SALES_CH,
        "feature_cols": FEATURE_COLS,
        "stage": stage,
        "trial_number": trial.number,
    }

    best_val = float('inf')
    for ep in range(max_epochs):
        train_one_epoch_origci(model, tr_loader, optimizer, criterion, device, grad_clip=1.0)
        val_loss = evaluate_origci(model, va_loader, criterion, device)
        scheduler.step()
        trial.report(val_loss, ep)

        if val_loss < best_val:
            best_val = val_loss
            # save checkpoint with loss value
            torch.save({
                "model_state": model.state_dict(),
                "best_val": best_val,
                "epoch": ep,
                "trial_params": trial.params,
                "loss_components": {"mae_weight": a, "mse_weight": b, "smape_weight": c}
            }, ckpt_path)
            
            # config에 loss value도 포함
            cfg_with_params = {**cfg, **trial.params, "best_val_loss": best_val}
            with open(cfg_path, "w") as f:
                json.dump(cfg_with_params, f, indent=2)
            
            trial.set_user_attr("ckpt_path", str(ckpt_path))
            trial.set_user_attr("cfg_path", str(cfg_path))
            trial.set_user_attr("best_val_loss", best_val)

        if trial.should_prune(): raise optuna.TrialPruned()
        stopper.step(val_loss)
        if stopper.stop: break

    return best_val

In [40]:
# === 2-Stage Optuna 실행 코드 ===
def run_two_stage_optuna():
    import torch, optuna
    from functools import partial
    
    SEED = 42
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)
    
    # === Stage 1: 빠른 스크리닝 ===
    print("=== Stage 1: Fast Screening ===")
    sampler1 = optuna.samplers.TPESampler(seed=SEED)
    pruner1 = optuna.pruners.MedianPruner(n_warmup_steps=3, n_min_trials=10)  # 강한 pruner
    study1 = optuna.create_study(direction="minimize", sampler=sampler1, pruner=pruner1)
    
    obj1 = partial(
        objective_origci_enhanced,
        device=device,
        lookback=CFG.lookback,
        horizon=CFG.horizon,
        c_in_raw=len(FEATURE_COLS),
        dataset=full_dataset,
        folds=folds,
        stage="stage1"
    )
    
    # 150-250 trials for screening
    study1.optimize(obj1, n_trials=20, show_progress_bar=True)
    
    # 상위 10개 trial 선택
    top_trials = sorted(study1.trials, key=lambda t: t.value if t.value is not None else float('inf'))[:10]
    print(f"\nStage 1 완료. 상위 10개 trial 선택됨")
    for i, t in enumerate(top_trials):
        print(f"  #{i+1}: Trial {t.number}, Loss: {t.value:.4f}")
    
    # === Stage 2: 정밀 재학습 ===
    print("\n=== Stage 2: Precise Re-training ===")
    sampler2 = optuna.samplers.TPESampler(seed=SEED+1)
    pruner2 = optuna.pruners.NopPruner()  # No pruning for precise training
    study2 = optuna.create_study(direction="minimize", sampler=sampler2, pruner=pruner2)
    
    # 상위 10개 설정을 study2에 enqueue
    for trial in top_trials:
        study2.enqueue_trial(trial.params)
    
    obj2 = partial(
        objective_origci_enhanced,
        device=device,
        lookback=CFG.lookback,
        horizon=CFG.horizon,
        c_in_raw=len(FEATURE_COLS),
        dataset=full_dataset,
        folds=folds,
        stage="stage2"
    )
    
    study2.optimize(obj2, n_trials=10, show_progress_bar=True)
    
    # 최종 상위 3개 선택
    final_top3 = sorted(study2.trials, key=lambda t: t.value if t.value is not None else float('inf'))[:3]
    print(f"\nStage 2 완료. 최종 상위 3개 trial:")
    for i, t in enumerate(final_top3):
        print(f"  #{i+1}: Trial {t.number}, Loss: {t.value:.4f}")
    
    return study1, study2, final_top3


In [41]:
# === 앙상블 추론 코드 ===

def load_model_from_trial_config(trial_info, device):
    """trial로부터 모델을 로드"""
    cfg_path = Path(trial_info.user_attrs["cfg_path"])
    ckpt_path = Path(trial_info.user_attrs["ckpt_path"])
    
    with open(cfg_path, "r") as f:
        cfg = json.load(f)
    
    # 모델 파라미터 추출
    ALLOWED = {"lookback","horizon","c_in","d_model","n_heads","depth",
               "patch_len","stride","dropout","sales_ch","target_channels"}
    cfg["c_in"] = cfg.get("c_in", len(cfg.get("feature_cols", FEATURE_COLS)))
    cfg["sales_ch"] = cfg.get("sales_ch", 0)
    cfg["target_channels"] = cfg.get("target_channels", [0])
    model_kwargs = {k: cfg[k] for k in cfg if k in ALLOWED}
    
    model = PatchTST_OrigCI(**model_kwargs).to(device)
    
    # 체크포인트 로드
    obj = torch.load(ckpt_path, map_location=device)
    if isinstance(obj, dict) and "model_state" in obj:
        sd = obj["model_state"]
    else:
        sd = obj
    model.load_state_dict(sd, strict=True)
    model.eval()
    
    return model, cfg

@torch.no_grad()
def ensemble_predict_test_file(models_and_configs: list, test_path: str, 
                              ensemble_method: str = "average") -> pd.DataFrame:
    """여러 모델로 앙상블 예측"""
    
    # 테스트 데이터 준비
    df = pd.read_csv(test_path)
    df = ensure_store_menu(df)
    df = df.sort_values(['store_menu','date']).reset_index(drop=True)
    df = add_calendar_features(df)
    
    # 모든 모델이 같은 lookback/horizon을 가진다고 가정
    L = models_and_configs[0][1]["lookback"]
    H = models_and_configs[0][1]["horizon"]
    feature_cols = models_and_configs[0][1]["feature_cols"]
    
    last_date = pd.to_datetime(df['date']).max()
    target_dates = pd.date_range(last_date + pd.Timedelta(days=1), periods=H, freq='D')
    
    # 각 store_menu별로 앙상블 예측
    ensemble_preds = {}
    
    for sm, g in df.groupby('store_menu', sort=False):
        assert len(g) >= L, f"{sm}: need >= {L} rows"
        X = build_input_tensor_from_block(g, feature_cols, L)
        
        # 모든 모델의 예측 수집
        model_predictions = []
        for model, cfg in models_and_configs:
            X_device = X.to(model.training)  # 모델이 있는 device로
            y_n, stats = model(X_device)
            if y_n.dim() == 3 and y_n.size(1) == 1:
                y_n = y_n.squeeze(1)
            y = model.revin(y_n, stats=stats, mode='denorm')
            y = torch.clamp(y, min=0.0).squeeze(0).cpu().numpy()
            model_predictions.append(y)
        
        # 앙상블 방법 선택
        if ensemble_method == "average":
            ensemble_pred = np.mean(model_predictions, axis=0)
        elif ensemble_method == "median":
            ensemble_pred = np.median(model_predictions, axis=0)
        elif ensemble_method == "voting":  # 각 timestep별 최빈값 (연속값이므로 average와 유사)
            ensemble_pred = np.mean(model_predictions, axis=0)
        else:
            raise ValueError(f"Unknown ensemble method: {ensemble_method}")
        
        ensemble_preds[sm] = ensemble_pred
    
    # 결과 DataFrame 구성
    sm_list = list(df['store_menu'].drop_duplicates())
    sub = pd.DataFrame(index=target_dates, columns=sm_list, dtype=float)
    for sm in sm_list:
        sub[sm] = ensemble_preds[sm]
    sub.index.name = 'date'
    
    return sub


def create_ensemble_submission(final_top3_trials, test_files, device):
    """상위 3개 모델로 앙상블 제출 파일 생성"""
    
    # 상위 3개 모델 로드
    models_and_configs = []
    print("Loading top 3 models for ensemble...")
    for i, trial in enumerate(final_top3_trials):
        print(f"  Loading model {i+1}/3 (Trial {trial.number}, Loss: {trial.value:.4f})")
        model, cfg = load_model_from_trial_config(trial, device)
        models_and_configs.append((model, cfg))
    
    # 각 TEST 파일에 대해 앙상블 예측
    print("Running ensemble predictions...")
    ensemble_subs = []
    for test_path in test_files:
        sub = ensemble_predict_test_file(models_and_configs, test_path, ensemble_method="average")
        ensemble_subs.append(sub)
    
    # 결과 병합
    merged = pd.concat(ensemble_subs, axis=0, join="outer")
    merged = merged[~merged.index.duplicated(keep="last")].sort_index()
    
    return merged

In [42]:

def main_two_stage_pipeline():
    # 2-stage optuna 실행
    study1, study2, final_top3 = run_two_stage_optuna()
    
    # 모든 trial 결과 요약 저장
    stage1_results = []
    for trial in study1.trials:
        if trial.value is not None:
            result = {
                "trial_number": trial.number,
                "stage": "stage1", 
                "objective_value": trial.value,
                "params": trial.params,
                "state": trial.state.name
            }
            if hasattr(trial, 'user_attrs') and trial.user_attrs:
                result.update(trial.user_attrs)
            stage1_results.append(result)
    
    stage2_results = []
    for trial in study2.trials:
        if trial.value is not None:
            result = {
                "trial_number": trial.number,
                "stage": "stage2",
                "objective_value": trial.value, 
                "params": trial.params,
                "state": trial.state.name
            }
            if hasattr(trial, 'user_attrs') and trial.user_attrs:
                result.update(trial.user_attrs)
            stage2_results.append(result)
    
    # 결과 저장
    results_dir = ART_DIR / "optimization_results"
    results_dir.mkdir(exist_ok=True, parents=True)
    
    pd.DataFrame(stage1_results).to_csv(results_dir / "stage1_all_trials.csv", index=False)
    pd.DataFrame(stage2_results).to_csv(results_dir / "stage2_all_trials.csv", index=False) 
    
    with open(results_dir / "final_top3_summary.json", "w") as f:
        top3_summary = []
        for i, trial in enumerate(final_top3):
            top3_summary.append({
                "rank": i+1,
                "trial_number": trial.number,
                "objective_value": trial.value,
                "params": trial.params
            })
        json.dump(top3_summary, f, indent=2)
    
    print(f"Optimization results saved to {results_dir}")
    
    # 앙상블 예측 데이터프레임 생성 및 저장
    test_files = sorted(glob.glob("./dataset/TEST_0*.csv"))
    if test_files:
        print("Creating ensemble predictions...")
        ensemble_sub = create_ensemble_submission(final_top3, test_files, device)
        
        # 음수 및 1 미만 값을 1로 클리핑
        ensemble_sub = ensemble_sub.clip(lower=1.0)
        
        # 원본 DataFrame 그대로 저장 (date가 index)
        ensemble_path = "./ensemble_predictions_top3.csv"
        ensemble_sub.to_csv(ensemble_path, encoding="utf-8-sig")
        print(f"Ensemble predictions saved: {ensemble_path}, shape={ensemble_sub.shape}")
        print(f"Date range: {ensemble_sub.index.min()} to {ensemble_sub.index.max()}")
        print(f"Store-menu count: {len(ensemble_sub.columns)}")
    
    return study1, study2, final_top3

In [43]:
# === 실행 ===
if __name__ == "__main__":
    # 기존의 study.optimize 부분을 다음으로 대체
    study1, study2, final_top3 = main_two_stage_pipeline()

[I 2025-08-18 17:58:16,942] A new study created in memory with name: no-name-cb6298ac-5c99-4660-b0d2-14a948e5b14d


Device: cuda
=== Stage 1: Fast Screening ===


Best trial: 0. Best value: 181.097:   5%|▌         | 1/20 [08:15<2:36:59, 495.78s/it]

[I 2025-08-18 18:06:32,715] Trial 0 finished with value: 181.09660041313572 and parameters: {'w_mae': 0.3745401188473625, 'w_mse': 0.9507143064099162, 'w_smape': 0.7319939418114051, 'smape_delta': 0.058006322999333615, 'smape_eps': 0.0002636875533972306, 'batch_size': 512, 'epochs': 21, 'd_model': 256, 'n_heads': 8, 'depth': 3, 'patch_len': 7, 'stride': 2, 'dropout': 0.15427033152408348, 'lr': 0.000750011895041699, 'wd_is_zero': False, 'weight_decay_pos': 7.122305833333853e-08}. Best is trial 0 with value: 181.09660041313572.


Best trial: 0. Best value: 181.097:  10%|█         | 2/20 [11:32<1:36:01, 320.08s/it]

...

Best trial: 0. Best value: 6.45151:  90%|█████████ | 9/10 [1:01:25<06:50, 410.41s/it]/home/wonjun/.conda/envs/wonjun_base/lib/python3.10/site-packages/optuna/trial/_trial.py:650: UserWarning: Fixed parameter 'epochs' with value 17 is out of range for distribution IntDistribution(high=140, log=False, low=70, step=1).
  warnings.warn(


[I 2025-08-18 20:26:04,773] Trial 8 finished with value: 68.8807787198247 and parameters: {'w_mae': 0.31692200515627766, 'w_mse': 0.1694927466860925, 'w_smape': 0.5568012624583502, 'smape_delta': 0.23099085509794084, 'smape_eps': 0.007560726762413614, 'batch_size': 512, 'epochs': 25, 'd_model': 256, 'n_heads': 4, 'depth': 4, 'patch_len': 7, 'stride': 4, 'dropout': 0.2670016025452699, 'lr': 0.0003156892767756196, 'wd_is_zero': True}. Best is trial 0 with value: 6.45151186558206.


Best trial: 0. Best value: 6.45151: 100%|██████████| 10/10 [1:05:23<00:00, 392.34s/it]

[I 2025-08-18 20:30:02,207] Trial 9 finished with value: 90.00412974097324 and parameters: {'w_mae': 0.7282163486118596, 'w_mse': 0.3677831327192532, 'w_smape': 0.6323058305935795, 'smape_delta': 0.06690855515432795, 'smape_eps': 0.002792799778748392, 'batch_size': 256, 'epochs': 17, 'd_model': 256, 'n_heads': 4, 'depth': 4, 'patch_len': 6, 'stride': 1, 'dropout': 0.24516666006036475, 'lr': 0.000660843951593475, 'wd_is_zero': True}. Best is trial 0 with value: 6.45151186558206.

Stage 2 완료. 최종 상위 3개 trial:
  #1: Trial 0, Loss: 6.4515
  #2: Trial 1, Loss: 6.4964
  #3: Trial 2, Loss: 7.8811
Optimization results saved to optuna_trial_results/optimization_results


NameError: name 'glob' is not defined

In [50]:
# === Merge per-TEST prediction CSVs into one submission CSV ===
# 요구:
# 1) ./result 폴더에 TEST별 예측 CSV가 존재(wide 혹은 long 가능)
# 2) ./result/sample_submission.csv 존재(날짜 열만 사용)
# 3) 후처리 없음. 단순 병합만 수행

from pathlib import Path
import pandas as pd

RESULT_DIR = Path("./result")
SUBMISSION_TEMPLATE = RESULT_DIR / "sample_submission_date.csv"
OUTPUT_CSV = RESULT_DIR / "submission_merged.csv"   # 원하는 이름으로 변경 가능
# 예측 CSV 자동 탐지 규칙: result 폴더의 csv 중 sample_submission 및 최종 제출 파일 제외
EXCLUDE_NAMES = {SUBMISSION_TEMPLATE.name, OUTPUT_CSV.name}

def is_candidate_csv(path: Path) -> bool:
    if path.name in EXCLUDE_NAMES:
        return False
    if "submission" in path.stem.lower() and "sample" not in path.stem.lower():
        # 기존에 만든 제출파일은 제외
        return False
    return path.suffix.lower() == ".csv"

def load_pred_csv(fp: Path) -> pd.DataFrame:
    """wide(date + 여러 store_menu 열)면 그대로 사용.
       long(date, store_menu, pred/forecast)면 wide로 pivot."""
    df = pd.read_csv(fp)
    # date 열 이름 정규화
    date_col = None
    for c in df.columns:
        if c.lower() == "date":
            date_col = c
            break
    if date_col is None:
        raise ValueError(f"{fp.name}: 'date' 열이 필요합니다.")
    # long 형태 감지
    lower_cols = {c.lower() for c in df.columns}
    if {"store_menu", "pred"}.issubset(lower_cols) or \
       {"store_menu", "forecast"}.issubset(lower_cols):
        # 열 이름 매핑
        sm_col = [c for c in df.columns if c.lower()=="store_menu"][0]
        y_col = "pred" if "pred" in lower_cols else "forecast"
        y_col = [c for c in df.columns if c.lower()==y_col][0]
        w = df.pivot_table(index=date_col, columns=sm_col, values=y_col, aggfunc="first").reset_index()
        w = w.sort_values(date_col).reset_index(drop=True)
        return w
    else:
        # wide 형태 가정: date + 여러 store_menu 열
        if df.columns[0].lower() != "date":
            # date를 맨 앞으로
            cols = list(df.columns)
            cols.remove(date_col)
            df = df[[date_col] + cols]
        df = df.sort_values(date_col).reset_index(drop=True)
        return df

# 1) 템플릿 날짜 로드
tmpl = pd.read_csv(SUBMISSION_TEMPLATE)
if tmpl.columns[0].lower() != "date":
    raise ValueError("sample_submission.csv의 첫 열이 'date' 여야 합니다.")
target_dates = tmpl.iloc[:, 0].astype(str).tolist()

# 2) 후보 예측 CSV 수집 및 로드
csv_files = sorted([p for p in RESULT_DIR.glob("*.csv") if is_candidate_csv(p)])
if not csv_files:
    raise FileNotFoundError("병합할 예측 CSV를 찾지 못했습니다. RESULT_DIR 내용을 확인하세요.")

parts = []
for fp in csv_files:
    try:
        dfp = load_pred_csv(fp)
        parts.append((dfp[ dfp.columns[0] ].min(), dfp))  # 정렬용 키(date 최소값)
    except Exception as e:
        print(f"[SKIP] {fp.name}: {e}")

if not parts:
    raise RuntimeError("유효한 예측 CSV가 없습니다.")

# 3) 파일 간 시간순으로 정렬 후 세로 병합
parts.sort(key=lambda x: x[0])
merged = pd.concat([p[1] for p in parts], axis=0, ignore_index=True)

# 4) 길이 검증 및 날짜 치환
if len(merged) != len(target_dates):
    print(f"[경고] 병합된 행수={len(merged)}, 템플릿 행수={len(target_dates)}. 불일치.")
# 날짜 열을 템플릿으로 교체
merged.iloc[:, 0] = target_dates[:len(merged)]

# 5) 저장
merged.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
print(f"Saved: {OUTPUT_CSV.resolve()}")


[SKIP] tft_final_1.csv: tft_final_1.csv: 'date' 열이 필요합니다.
[SKIP] tft_raw_1.csv: tft_raw_1.csv: 'date' 열이 필요합니다.
Saved: /home/wonjun/Aimers/result/submission_merged.csv
